# 07 — Eight-Fold Signer-Independent Preparation

**Goal.** Create the **single frozen experiment manifest** that every model notebook reads: 3 models × 2 branches × 8 folds, all on identical splits.

**Design**
- 8 folds, each with **12 train / 2 validation / 2 test signers**, all disjoint. Fold *i* tests pair *i* and validates on pair *(i+1) mod 8*, so every signer is a test signer exactly once and a validation signer exactly once.
- Pooling the 8 test sets gives **exactly 6,000 out-of-fold predictions**, each from a model that never saw that clip's signer. This is the primary result used in 13.

**Key design decisions**
- **Pairs are optimised, not arbitrary.** Signers differ a lot (mean length 66–203 frames; left-hand detection 0.56–0.97). Pairing two hard signers inflates fold variance for reasons unrelated to the face question. Pairs therefore minimise the variance of the summed z-scores across pairs, weighted 50% sequence length and 50% hand-tracking quality.
- The objective decomposes over pairs, so it is solved **exactly** with bitmask DP over 2¹⁶ states. It is deterministic and uses signer metadata only, never model results.
- Clip count is reported but not optimised (it is near-binary). The constraint *p2 and p3 are never paired* removes the only imbalance that matters.
- **One signer-overlap control** (a random label-stratified 4,500 / 750 / 750 split ignoring signer) measures how much signer overlap inflates scores. One split rather than eight gives the same contrast in 6 runs instead of 48. It is reported as a *gap*, discussed as likely memorisation, and not asserted to be identity leakage.

**Note.** `master_manifest.csv` stores absolute paths from this runtime. Downstream notebooks rebuild paths from `relative_path` after verifying the experiment ID and hashes.

**Inputs → Outputs.** 06 output + both 05 branches → `experiment_manifest.json`, `master_manifest.csv`, `fold_assignments.csv`, `CONTROL_signer_overlap_split.csv`, `signer_pairs.json`, `label_map.json`, `fold_summary.csv`.

In [ ]:
# ============================================================
# CELL 1 — Configuration
# ============================================================
from __future__ import annotations

import hashlib
import itertools
import json
import os
import platform
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MANIFEST_VERSION = "nb7-v5"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

N_FOLDS = 8
TEST_SIGNERS_PER_FOLD = 2
VALIDATION_SIGNERS_PER_FOLD = 2
EXPECTED_CLIPS = 6000
EXPECTED_LABELS = 200
EXPECTED_SIGNERS = 16
EXPECTED_PIPELINE_ID = "24c9d5ccb3f1"

# Feature widths this experiment is defined to have. Checked against the
# statistics files so a file from a different preprocessing run is rejected.
EXPECTED_BRANCH_WIDTHS = {"full": 1759, "reduced": 631}

# Signers that must not share a pair. p2 and p3 hold 200 clips each rather than
# 400, so pairing them would make one test fold a third smaller than the rest.
FORBIDDEN_PAIRS: List[frozenset] = [frozenset({"p2", "p3"})]

# Pre-committed pairing objective: half the weight on sequence length, half on
# overall hand-tracking quality. Both are measured properties of the data,
# recorded before any model exists.
#
# Clip count is deliberately NOT an optimisation feature. It is near-binary
# (200 for p2 and p3, 400 for everyone else), so standardising it would hand
# two signers enormous leverage over the objective. The p2/p3 constraint below
# already prevents the only clip-count imbalance that matters. Clip count is
# reported for every pair but does not steer the search.
BALANCE_WEIGHTS: Dict[str, float] = {"mean_frames": 0.5, "hand_quality": 0.5}
REPORTING_FEATURES = ["clips", "mean_frames", "left_hand_rate", "right_hand_rate",
                      "hand_quality"]

OVERRIDE_AUDIT_DIR: Optional[str] = None

WORKING = Path("/kaggle/working")
PLOT_DIR = WORKING / "fold_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_MANIFEST = WORKING / "master_manifest.csv"
FOLD_ASSIGNMENTS = WORKING / "fold_assignments.csv"
SIGNER_PAIRS = WORKING / "signer_pairs.json"
LABEL_MAP = WORKING / "label_map.json"
FOLD_SUMMARY = WORKING / "fold_summary.csv"
EXPERIMENT_MANIFEST = WORKING / "experiment_manifest.json"
CONTROL_ASSIGNMENTS = WORKING / "CONTROL_signer_overlap_split.csv"

ENVIRONMENT = {
    "manifest_version": MANIFEST_VERSION, "python": platform.python_version(),
    "numpy": np.__version__, "pandas": pd.__version__, "seed": RANDOM_SEED,
}


def atomic_write(path: Path, writer) -> None:
    temporary = path.parent / (path.name + ".tmp")
    with open(temporary, "wb") as handle:
        writer(handle)
    os.replace(temporary, path)


print("Environment")
for key, value in ENVIRONMENT.items():
    print(f"  {key}: {value}")
print(f"\nDesign: {N_FOLDS} folds | {EXPECTED_SIGNERS - TEST_SIGNERS_PER_FOLD - VALIDATION_SIGNERS_PER_FOLD} train / "
      f"{VALIDATION_SIGNERS_PER_FOLD} val / {TEST_SIGNERS_PER_FOLD} test signers")

In [ ]:
# ============================================================
# CELL 2 — Load the audit and refuse anything that did not PASS
# ============================================================

def find_one(pattern: str, description: str, override: Optional[str] = None) -> Path:
    if override:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(f"Override not found: {path}")
        return path
    root = Path("/kaggle/input")
    if not root.exists():
        raise FileNotFoundError("/kaggle/input missing. Attach the notebook 6 output.")
    hits = sorted(p for p in root.glob("**/" + pattern) if p.is_file())
    if not hits:
        raise FileNotFoundError(
            f"Could not find {description} ({pattern}). Attach the notebook 6 output.")
    if len(hits) > 1:
        listing = "\n".join(f"    {h}" for h in hits)
        raise RuntimeError(f"{len(hits)} copies of {pattern}; refusing to guess.\n{listing}")
    return hits[0]


AUDIT_JSON_PATH = find_one("post_normalization_audit.json", "the audit summary",
                           f"{OVERRIDE_AUDIT_DIR}/post_normalization_audit.json"
                           if OVERRIDE_AUDIT_DIR else None)
AUDIT_CSV_PATH = find_one("post_normalization_audit.csv", "the per-clip audit table",
                          f"{OVERRIDE_AUDIT_DIR}/post_normalization_audit.csv"
                          if OVERRIDE_AUDIT_DIR else None)
SIGNER_SUMMARY_PATH = find_one("signer_quality_summary.csv", "the per-signer summary",
                               f"{OVERRIDE_AUDIT_DIR}/signer_quality_summary.csv"
                               if OVERRIDE_AUDIT_DIR else None)

audit_summary = json.loads(AUDIT_JSON_PATH.read_text(encoding="utf-8"))
print(f"audit summary : {AUDIT_JSON_PATH}")
print(f"audit table   : {AUDIT_CSV_PATH}")
print(f"signer summary: {SIGNER_SUMMARY_PATH}")

AUDIT_VERDICT = audit_summary.get("verdict")
AUDIT_MODE = audit_summary.get("mode")
CLIP_TABLE_HASH = audit_summary.get("clip_table_hash")
PIPELINE_ID = audit_summary.get("pipeline_id")

print(f"\naudit verdict : {AUDIT_VERDICT}")
print(f"audit mode    : {AUDIT_MODE}")
print(f"clip hash     : {CLIP_TABLE_HASH}")
print(f"pipeline id   : {PIPELINE_ID}")

if AUDIT_VERDICT != "PASS":
    raise RuntimeError(
        f"The audit verdict is {AUDIT_VERDICT}, not PASS. Every fold defined here would "
        "rest on data the audit rejected. Fix notebook 6 first.")
if AUDIT_MODE != "full":
    raise RuntimeError(
        f"The audit ran in '{AUDIT_MODE}' mode, not 'full'. A smoke-test pass covers only a "
        "subset of clips and is not permission to build the experiment manifest.")
if PIPELINE_ID != EXPECTED_PIPELINE_ID:
    raise RuntimeError(f"pipeline id {PIPELINE_ID} != {EXPECTED_PIPELINE_ID}")

# ---- Statistics files: full provenance check ----------------------------
# Notebook 8 standardises with these, so every identifying field is checked.
# Verifying only the PASS stamp would still accept a file from a different
# preprocessing run that also happened to pass its own audit.
AUDIT_SUCCESSFUL_CLIP_HASH = audit_summary.get("successful_clip_id_hash")
AUDIT_SUCCESSFUL_CLIP_COUNT = int(audit_summary.get("counts", {}).get("clips_audited", 0))

STATS_PATHS: Dict[str, Path] = {}
STATS_PROVENANCE: Dict[str, Dict[str, Any]] = {}
for branch in ("full", "reduced"):
    STATS_PATHS[branch] = find_one(f"signer_feature_stats_{branch}.npz",
                                   f"{branch} per-signer statistics")
    with np.load(STATS_PATHS[branch], allow_pickle=False) as archive:
        def field(name: str) -> Any:
            key = f"__{name}__"
            if key not in archive.files:
                raise RuntimeError(
                    f"{branch} statistics lack '{key}'. They were written by an older "
                    "audit; rerun notebook 6.")
            return archive[key][0]

        found = {
            "branch": str(field("branch")),
            "audit_verdict": str(field("audit_verdict")),
            "pipeline_id": str(field("pipeline_id")),
            "clip_table_hash": str(field("clip_table_hash")),
            "successful_clip_id_hash": str(field("successful_clip_id_hash")),
            "successful_clip_count": int(field("successful_clip_count")),
            "width": int(field("width")),
            "audit_version": str(field("audit_version")),
        }
        expected = {
            "branch": branch,
            "audit_verdict": "PASS",
            "pipeline_id": EXPECTED_PIPELINE_ID,
            "clip_table_hash": CLIP_TABLE_HASH,
            "successful_clip_id_hash": AUDIT_SUCCESSFUL_CLIP_HASH,
            "successful_clip_count": AUDIT_SUCCESSFUL_CLIP_COUNT,
            "width": EXPECTED_BRANCH_WIDTHS[branch],
        }
        mismatched = {key: (found[key], value) for key, value in expected.items()
                      if value is not None and found[key] != value}
        if mismatched:
            detail = "; ".join(f"{k}: found {f!r}, expected {e!r}"
                               for k, (f, e) in mismatched.items())
            raise RuntimeError(
                f"{branch} statistics do not belong to this experiment. {detail}")
        STATS_PROVENANCE[branch] = found

    print(f"  {branch} statistics verified: {STATS_PATHS[branch].name}")
    print(f"    branch={found['branch']} width={found['width']} "
          f"clips={found['successful_clip_count']:,} audit={found['audit_version']}")

# ---- Phase 5 landmark roots, so manifest paths can be checked ------------
BRANCH_ROOTS: Dict[str, Path] = {}
for branch in ("full", "reduced"):
    hits = [p for p in Path("/kaggle/input").glob(f"**/normalized_landmarks/{branch}") if p.is_dir()]
    if not hits:
        raise FileNotFoundError(
            f"No normalized_landmarks/{branch} found. Attach the Phase 5 '{branch}' output "
            "so the manifest paths can be verified to exist.")
    if len(hits) > 1:
        listing = "\n".join(f"    {h}" for h in hits)
        raise RuntimeError(f"{len(hits)} '{branch}' landmark roots attached.\n{listing}")
    BRANCH_ROOTS[branch] = hits[0]
    print(f"  {branch} landmarks: {hits[0]}")

audit = pd.read_csv(AUDIT_CSV_PATH)
signer_summary = pd.read_csv(SIGNER_SUMMARY_PATH)
print(f"\naudit rows: {len(audit):,} | signer rows: {len(signer_summary):,}")

In [ ]:
# ============================================================
# CELL 3 — Master manifest and label map
# ============================================================
ok = audit[audit["status"] == "ok"].copy()
if len(ok) != EXPECTED_CLIPS:
    raise RuntimeError(f"{len(ok):,} successful audit rows, expected {EXPECTED_CLIPS:,}")

labels = sorted(ok["label"].astype(str).unique())
if len(labels) != EXPECTED_LABELS:
    raise RuntimeError(f"{len(labels)} labels, expected {EXPECTED_LABELS}")

# One shared label map for all 48 runs. Alphabetical, so it is reproducible
# from the data alone and never depends on file order.
LABEL_TO_ID = {label: index for index, label in enumerate(labels)}
ID_TO_LABEL = {index: label for label, index in LABEL_TO_ID.items()}

manifest = pd.DataFrame({
    "clip_id": ok["clip"].astype(str),
    "label": ok["label"].astype(str),
    "label_id": ok["label"].astype(str).map(LABEL_TO_ID).astype(int),
    "signer_id": ok["signer_id"].astype(str),
    "session": ok["session"].astype(str),
    "relative_path": ok["relative_path"].astype(str),
    "original_frames": ok["full_frames"].astype(int),
    "face_detection_rate": ok["computed_face_rate"].astype(float),
    "pose_detection_rate": ok["computed_pose_rate"].astype(float),
    "left_hand_detection_rate": ok["computed_left_hand_rate"].astype(float),
    "right_hand_detection_rate": ok["computed_right_hand_rate"].astype(float),
})
manifest["full_path"] = manifest["relative_path"].map(lambda r: str(BRANCH_ROOTS["full"] / r))
manifest["reduced_path"] = manifest["relative_path"].map(
    lambda r: str(BRANCH_ROOTS["reduced"] / r))
# The single tracking-quality score the pairing objective uses.
manifest["hand_quality"] = (manifest["left_hand_detection_rate"]
                            + manifest["right_hand_detection_rate"]) / 2
manifest = manifest.sort_values("clip_id", kind="mergesort").reset_index(drop=True)

missing_full = [p for p in manifest["full_path"] if not Path(p).exists()]
missing_reduced = [p for p in manifest["reduced_path"] if not Path(p).exists()]
print(f"manifest rows       : {len(manifest):,}")
print(f"missing full paths  : {len(missing_full):,}")
print(f"missing reduced     : {len(missing_reduced):,}")
if missing_full or missing_reduced:
    raise FileNotFoundError(
        f"{len(missing_full)} full and {len(missing_reduced)} reduced paths do not exist. "
        f"First missing: {(missing_full + missing_reduced)[0]}")

print(f"\nunique clip ids     : {manifest['clip_id'].nunique():,}")
print(f"labels              : {manifest['label'].nunique()}")
print(f"signers             : {manifest['signer_id'].nunique()}")
print(f"label_id range      : {manifest['label_id'].min()}..{manifest['label_id'].max()}")
print(f"\nfirst labels: {labels[:6]}")
print(f"last labels : {labels[-3:]}")

coverage = manifest.groupby("signer_id")["label"].nunique()
print(f"\nclasses per signer  : min {coverage.min()}, max {coverage.max()}")
if coverage.min() != EXPECTED_LABELS:
    raise RuntimeError("Not every signer covers every class; the fold design assumes they do.")

In [ ]:
# ============================================================
# CELL 4 — Deterministic signer pairing, solved exactly
# ============================================================
signer_table = manifest.groupby("signer_id").agg(
    clips=("clip_id", "size"),
    mean_frames=("original_frames", "mean"),
    left_hand_rate=("left_hand_detection_rate", "mean"),
    right_hand_rate=("right_hand_detection_rate", "mean"),
    face_rate=("face_detection_rate", "mean"),
    pose_rate=("pose_detection_rate", "mean"),
).reset_index()
# The single tracking-quality score the objective balances.
signer_table["hand_quality"] = (signer_table["left_hand_rate"]
                                + signer_table["right_hand_rate"]) / 2
signer_table["signer_number"] = signer_table["signer_id"].str.lstrip("p").astype(int)
signer_table = signer_table.sort_values("signer_number").reset_index(drop=True)

print("SIGNER PROPERTIES (from the audit, before any model exists)")
print(signer_table.round(4).to_string(index=False))


def pair_signers(table: pd.DataFrame, weights: Dict[str, float],
                 forbidden: Sequence[frozenset]) -> List[Tuple[str, str]]:
    """Exact minimum-variance pairing by bitmask dynamic programming.

    Cost of a pair is the weighted squared distance of its summed z-scores from
    zero. Every signer is used exactly once, so the mean pair-sum is fixed;
    minimising the total of those squared deviations is therefore precisely
    minimising the weighted variance across pairs. That objective decomposes
    over pairs, which is what allows an exact answer rather than a greedy
    approximation.
    """
    # Sort by numeric signer id here, so the result cannot depend on the row
    # order of the caller's frame. The self-test below still checks this, but
    # the guarantee is structural rather than merely observed.
    table = table.assign(
        _order=table["signer_id"].map(lambda s: int(str(s).lstrip("p")))
    ).sort_values("_order", kind="mergesort").drop(columns="_order").reset_index(drop=True)

    names = table["signer_id"].tolist()
    count = len(names)
    if count % 2:
        raise ValueError(f"{count} signers cannot be split into pairs")

    features = list(weights)
    weight_vector = np.array([weights[f] for f in features], dtype=float)
    values = table[features].to_numpy(dtype=float)
    spread = values.std(axis=0)
    spread[spread == 0] = 1.0
    z = (values - values.mean(axis=0)) / spread

    infinity = float("inf")
    cost = np.full((count, count), infinity)
    for i in range(count):
        for j in range(i + 1, count):
            if frozenset({names[i], names[j]}) in forbidden:
                continue
            cost[i, j] = cost[j, i] = float(np.sum(weight_vector * (z[i] + z[j]) ** 2))

    size = 1 << count
    best = np.full(size, infinity)
    choice = np.full(size, -1, dtype=np.int64)
    best[0] = 0.0
    for mask in range(size):
        if best[mask] == infinity:
            continue
        first = next((i for i in range(count) if not (mask >> i) & 1), -1)
        if first < 0:
            continue
        for j in range(first + 1, count):
            if (mask >> j) & 1 or cost[first, j] == infinity:
                continue
            following = mask | (1 << first) | (1 << j)
            candidate = best[mask] + cost[first, j]
            if candidate < best[following] - 1e-12:
                best[following] = candidate
                choice[following] = first * count + j

    if best[size - 1] == infinity:
        raise RuntimeError("No valid pairing exists under the forbidden-pair constraints")

    mask, pairs = size - 1, []
    while mask:
        i, j = divmod(int(choice[mask]), count)
        pairs.append((names[i], names[j]))
        mask &= ~((1 << i) | (1 << j))

    ordered = [tuple(sorted(pair, key=lambda s: int(str(s).lstrip("p")))) for pair in pairs]
    return sorted(ordered, key=lambda pair: int(str(pair[0]).lstrip("p")))


print(f"\nObjective: " + " + ".join(f"{w:.0%} {f}" for f, w in BALANCE_WEIGHTS.items()))
print(f"Reported but not optimised: clips")
SIGNER_PAIRS_LIST = pair_signers(signer_table, BALANCE_WEIGHTS, FORBIDDEN_PAIRS)

pair_rows = []
lookup = signer_table.set_index("signer_id")
for index, (first, second) in enumerate(SIGNER_PAIRS_LIST):
    pair_rows.append({
        "pair_index": index, "signer_a": first, "signer_b": second,
        "clips": int(lookup.loc[first, "clips"] + lookup.loc[second, "clips"]),
        "mean_frames": float((lookup.loc[first, "mean_frames"]
                              + lookup.loc[second, "mean_frames"]) / 2),
        "left_hand_rate": float((lookup.loc[first, "left_hand_rate"]
                                 + lookup.loc[second, "left_hand_rate"]) / 2),
        "right_hand_rate": float((lookup.loc[first, "right_hand_rate"]
                                  + lookup.loc[second, "right_hand_rate"]) / 2),
        "hand_quality": float((lookup.loc[first, "hand_quality"]
                               + lookup.loc[second, "hand_quality"]) / 2),
    })
pair_table = pd.DataFrame(pair_rows)

print("\nSIGNER PAIRS (each becomes one test fold)")
print(pair_table.round(4).to_string(index=False))

print("\nBALANCE: spread across pairs vs across individual signers")
for pair_column, signer_column in (("mean_frames", "mean_frames"),
                                   ("hand_quality", "hand_quality"),
                                   ("left_hand_rate", "left_hand_rate"),
                                   ("right_hand_rate", "right_hand_rate"),
                                   ("clips", "clips")):
    pair_std = pair_table[pair_column].std()
    signer_std = signer_table[signer_column].std()
    ratio = signer_std / pair_std if pair_std > 0 else float("inf")
    optimised = "  <- optimised" if pair_column in BALANCE_WEIGHTS else ""
    print(f"  {pair_column:<17} pairs std {pair_std:8.3f} | signers std {signer_std:8.3f} "
          f"| {ratio:.1f}x tighter{optimised}")

# ---- Self-tests -----------------------------------------------------------
_failures: List[str] = []


def _check(name: str, condition: bool, detail: str = "") -> None:
    print(f"{'PASS' if condition else 'FAIL'}  {name}  {detail}")
    if not condition:
        _failures.append(name)


print()
_flat = [signer for pair in SIGNER_PAIRS_LIST for signer in pair]
_check("every signer appears exactly once", sorted(_flat) == sorted(signer_table["signer_id"]))
_check(f"{N_FOLDS} pairs produced", len(SIGNER_PAIRS_LIST) == N_FOLDS, str(len(SIGNER_PAIRS_LIST)))
for forbidden in FORBIDDEN_PAIRS:
    _check(f"forbidden pair {sorted(forbidden)} not used",
           not any(frozenset(pair) == forbidden for pair in SIGNER_PAIRS_LIST))
_check("pairing is deterministic",
       pair_signers(signer_table, BALANCE_WEIGHTS, FORBIDDEN_PAIRS) == SIGNER_PAIRS_LIST)
_check("pairing ignores row order",
       pair_signers(signer_table.iloc[::-1].reset_index(drop=True),
                    BALANCE_WEIGHTS, FORBIDDEN_PAIRS) == SIGNER_PAIRS_LIST)
_check("pairing beats naive adjacent pairing",
       pair_table["mean_frames"].std()
       < signer_table.sort_values("mean_frames")["mean_frames"]
       .to_numpy().reshape(-1, 2).mean(axis=1).std(),
       "lower spread than pairing neighbours by length")

if _failures:
    raise RuntimeError(f"Pairing self-tests failed: {_failures}")
print("\nPairing self-tests clean.")

In [ ]:
# ============================================================
# CELL 5 — Optimise the cyclic order, then build the folds
# ============================================================
# Fold i tests final_cyclic_pairs[i] and validates final_cyclic_pairs[(i+1) mod 8],
# where final_cyclic_pairs is SIGNER_PAIRS_LIST reordered by PAIR_ORDER. Because the pairs
# partition the signers, every signer is a test signer exactly once and a
# validation signer exactly once, and the two roles never coincide.
#
# The ORDER matters even though pair membership does not change. Two pairs hold
# 600 clips (they contain p2 and p3); the rest hold 800. If the two small pairs
# sit next to each other in the cycle, one fold holds both out at once and
# trains on ~4,800 clips while others train on ~4,400. Separating them halves
# that spread. Test and validation clip counts are fixed multisets of pair
# sizes and so are unaffected by ordering; only the training size varies.

TOTAL_CLIPS = int(pair_table["clips"].sum())
PAIR_SIZES = pair_table["clips"].tolist()


def train_sizes(order: Sequence[int]) -> List[int]:
    n = len(order)
    return [TOTAL_CLIPS - PAIR_SIZES[order[i]] - PAIR_SIZES[order[(i + 1) % n]]
            for i in range(n)]


def optimise_cyclic_order(n: int) -> List[int]:
    """Exhaustive search over cyclic orders, minimising training-size variance.

    Rotations of a cycle are equivalent, so pair 0 is pinned first and the
    remaining (n-1)! orders are enumerated - 5,040 for n=8. Ties are broken
    lexicographically on the order tuple, which makes the answer unique.
    """
    best: Optional[Tuple[float, Tuple[int, ...]]] = None
    for rest in itertools.permutations(range(1, n)):
        order = (0,) + rest
        candidate = (round(float(np.var(train_sizes(order))), 9), order)
        if best is None or candidate < best:
            best = candidate
    return list(best[1])


baseline_order = list(range(N_FOLDS))
baseline_train = train_sizes(baseline_order)
PAIR_ORDER = optimise_cyclic_order(N_FOLDS)

# Reordering must never alter which signers are paired together.
if sorted(PAIR_ORDER) != list(range(N_FOLDS)):
    raise RuntimeError("The optimised order is not a permutation of the pairs.")
if {frozenset(SIGNER_PAIRS_LIST[i]) for i in PAIR_ORDER} != {frozenset(p) for p in SIGNER_PAIRS_LIST}:
    raise RuntimeError("Pair membership changed during order optimisation.")
optimised_train = train_sizes(PAIR_ORDER)

print("CYCLIC ORDER OPTIMISATION")
print(f"  before: order {baseline_order}")
print(f"          train sizes {baseline_train}")
print(f"          spread {max(baseline_train) - min(baseline_train)}, "
      f"std {np.std(baseline_train):.1f}")
print(f"  after : order {PAIR_ORDER}")
pair_names = [f"{row.signer_a}+{row.signer_b}" for row in pair_table.itertuples()]
print("          " + " -> ".join(pair_names[index] for index in PAIR_ORDER))
print(f"          train sizes {optimised_train}")
print(f"          spread {max(optimised_train) - min(optimised_train)}, "
      f"std {np.std(optimised_train):.1f}")

small_pairs = [index for index, size in enumerate(PAIR_SIZES) if size < max(PAIR_SIZES)]
positions = {pair_index: PAIR_ORDER.index(pair_index) for pair_index in small_pairs}
adjacent_small = False
for a, b in itertools.combinations(small_pairs, 2):
    distance = abs(positions[a] - positions[b])
    if min(distance, N_FOLDS - distance) == 1:
        adjacent_small = True
smaller_names = ", ".join(pair_names[index] for index in small_pairs)
print(f"\n  smaller pairs ({smaller_names}) at cycle positions "
      f"{sorted(positions.values())}")
print(f"  adjacent in the cycle (including the wrap-around): {adjacent_small}")

if adjacent_small:
    raise RuntimeError(
        "The two smaller pairs are adjacent in the optimised cycle, so one fold "
        "would hold both out at once. The order optimisation did not achieve its "
        "purpose; inspect PAIR_SIZES.")
if np.std(optimised_train) > np.std(baseline_train) + 1e-9:
    raise RuntimeError("Order optimisation increased the training-size spread.")

all_signers = sorted(signer_table["signer_id"], key=lambda s: int(str(s).lstrip("p")))
fold_definitions: List[Dict[str, Any]] = []

for fold in range(N_FOLDS):
    test_pair_index = PAIR_ORDER[fold]
    validation_pair_index = PAIR_ORDER[(fold + 1) % N_FOLDS]
    test_signers = list(SIGNER_PAIRS_LIST[test_pair_index])
    validation_signers = list(SIGNER_PAIRS_LIST[validation_pair_index])
    training_signers = [s for s in all_signers
                        if s not in test_signers and s not in validation_signers]
    fold_definitions.append({
        "fold": fold, "test_pair_index": test_pair_index,
        "validation_pair_index": validation_pair_index,
        "train_signers": training_signers,
        "validation_signers": validation_signers,
        "test_signers": test_signers,
    })

assignment_rows = []
for definition in fold_definitions:
    membership = {}
    for signer in definition["train_signers"]:
        membership[signer] = "train"
    for signer in definition["validation_signers"]:
        membership[signer] = "validation"
    for signer in definition["test_signers"]:
        membership[signer] = "test"
    split = manifest["signer_id"].map(membership)
    assignment_rows.append(pd.DataFrame({
        "fold": definition["fold"], "clip_id": manifest["clip_id"],
        "signer_id": manifest["signer_id"], "label_id": manifest["label_id"],
        "split": split,
    }))

fold_assignments = pd.concat(assignment_rows, ignore_index=True)
print(f"assignment rows: {len(fold_assignments):,} "
      f"({N_FOLDS} folds x {len(manifest):,} clips)")

# ---- Assertions -----------------------------------------------------------
problems: List[str] = []

test_counts = {signer: 0 for signer in all_signers}
validation_counts = {signer: 0 for signer in all_signers}
for definition in fold_definitions:
    for signer in definition["test_signers"]:
        test_counts[signer] += 1
    for signer in definition["validation_signers"]:
        validation_counts[signer] += 1

    partitions = [set(definition["train_signers"]), set(definition["validation_signers"]),
                  set(definition["test_signers"])]
    if set.union(*partitions) != set(all_signers):
        problems.append(f"fold {definition['fold']}: partitions do not cover all signers")
    for a in range(3):
        for b in range(a + 1, 3):
            if partitions[a] & partitions[b]:
                problems.append(f"fold {definition['fold']}: overlapping partitions "
                                f"{sorted(partitions[a] & partitions[b])}")
    if len(definition["train_signers"]) != EXPECTED_SIGNERS - 4:
        problems.append(f"fold {definition['fold']}: {len(definition['train_signers'])} "
                        f"train signers, expected {EXPECTED_SIGNERS - 4}")

for signer, count in test_counts.items():
    if count != 1:
        problems.append(f"{signer} is a test signer {count} times, expected once")
for signer, count in validation_counts.items():
    if count != 1:
        problems.append(f"{signer} is a validation signer {count} times, expected once")

if fold_assignments["split"].isna().any():
    problems.append("some clips were not assigned to any split")

per_fold_clips = fold_assignments.groupby("fold")["clip_id"].nunique()
if not (per_fold_clips == len(manifest)).all():
    problems.append("a fold does not contain every clip exactly once")

duplicated = fold_assignments.duplicated(subset=["fold", "clip_id"]).sum()
if duplicated:
    problems.append(f"{duplicated} clips appear in more than one split within a fold")

for fold in range(N_FOLDS):
    subset = fold_assignments[fold_assignments["fold"] == fold]
    for split in ("train", "validation", "test"):
        classes = subset.loc[subset["split"] == split, "label_id"].nunique()
        if classes != EXPECTED_LABELS:
            problems.append(f"fold {fold} {split}: {classes} classes, expected {EXPECTED_LABELS}")
    # The decisive leakage check, made on clip ids rather than signer names.
    clip_sets = {split: set(subset.loc[subset["split"] == split, "clip_id"])
                 for split in ("train", "validation", "test")}
    signer_sets = {split: set(manifest.loc[manifest["clip_id"].isin(clips_), "signer_id"])
                   for split, clips_ in clip_sets.items()}
    if signer_sets["train"] & signer_sets["test"]:
        problems.append(f"fold {fold}: signer leakage between train and test")
    if signer_sets["train"] & signer_sets["validation"]:
        problems.append(f"fold {fold}: signer leakage between train and validation")
    if signer_sets["validation"] & signer_sets["test"]:
        problems.append(f"fold {fold}: signer leakage between validation and test")

print("\nLEAKAGE AND STRUCTURE CHECKS")
if problems:
    for problem in problems:
        print(f"  FAIL  {problem}")
    raise RuntimeError(f"{len(problems)} fold problems found; the manifest is not usable.")
print(f"  PASS  every signer tested exactly once")
print(f"  PASS  every signer validated exactly once")
print(f"  PASS  train/validation/test disjoint in all {N_FOLDS} folds")
print(f"  PASS  all {EXPECTED_LABELS} classes present in every split of every fold")
print(f"  PASS  no clip in two splits of the same fold")

# p2 and p3 sit in different pairs; this confirms those pairs are never the
# test and validation pair of the same fold, which the cycle order guarantees.
small_signers = sorted({signer for pair in FORBIDDEN_PAIRS for signer in pair})
simultaneous = [d["fold"] for d in fold_definitions
                if set(small_signers) <= set(d["test_signers"]) | set(d["validation_signers"])]
if simultaneous:
    raise RuntimeError(f"{small_signers} are held out together in folds {simultaneous}")
print(f"  PASS  {small_signers} never held out together")

fold_clip_counts = pd.DataFrame([{
    "fold": d["fold"],
    "train": int((fold_assignments["fold"].eq(d["fold"])
                  & fold_assignments["split"].eq("train")).sum()),
    "validation": int((fold_assignments["fold"].eq(d["fold"])
                       & fold_assignments["split"].eq("validation")).sum()),
    "test": int((fold_assignments["fold"].eq(d["fold"])
                 & fold_assignments["split"].eq("test")).sum()),
} for d in fold_definitions])
TRAIN_SPREAD = {
    "min": int(fold_clip_counts["train"].min()), "max": int(fold_clip_counts["train"].max()),
    "std": float(fold_clip_counts["train"].std(ddof=0)),
    "baseline_min": int(min(baseline_train)), "baseline_max": int(max(baseline_train)),
    "baseline_std": float(np.std(baseline_train)),
}
if TRAIN_SPREAD["std"] > TRAIN_SPREAD["baseline_std"] + 1e-9:
    raise RuntimeError("Training-size spread is worse than the unoptimised order.")
print(f"  PASS  training-size spread minimised: {TRAIN_SPREAD['min']}-{TRAIN_SPREAD['max']} "
      f"(std {TRAIN_SPREAD['std']:.1f}) vs baseline {TRAIN_SPREAD['baseline_min']}-"
      f"{TRAIN_SPREAD['baseline_max']} (std {TRAIN_SPREAD['baseline_std']:.1f})")

for definition in fold_definitions:
    if len(definition["train_signers"]) != 12 or len(definition["validation_signers"]) != 2             or len(definition["test_signers"]) != 2:
        raise RuntimeError(f"fold {definition['fold']}: wrong partition sizes")
print("  PASS  every fold has 12 train / 2 validation / 2 test signers")

pooled = fold_assignments[fold_assignments["split"] == "test"]
print(f"\n  pooled out-of-fold test predictions: {len(pooled):,} "
      f"(unique clips {pooled['clip_id'].nunique():,})")
if pooled["clip_id"].nunique() != len(manifest) or len(pooled) != len(manifest):
    raise RuntimeError("The eight test sets do not partition the dataset exactly once.")

In [ ]:
# ============================================================
# CELL 6 — Fold summary, control split, plots
# ============================================================
summary_rows = []
for definition in fold_definitions:
    fold = definition["fold"]
    subset = fold_assignments[fold_assignments["fold"] == fold]
    row: Dict[str, Any] = {
        "fold": fold,
        "train_signers": " ".join(definition["train_signers"]),
        "validation_signers": " ".join(definition["validation_signers"]),
        "test_signers": " ".join(definition["test_signers"]),
    }
    for split in ("train", "validation", "test"):
        clip_ids = subset.loc[subset["split"] == split, "clip_id"]
        part = manifest[manifest["clip_id"].isin(set(clip_ids))]
        row[f"{split}_clips"] = len(part)
        row[f"{split}_classes"] = part["label"].nunique()
        row[f"{split}_mean_frames"] = round(float(part["original_frames"].mean()), 1)
        row[f"{split}_left_hand_rate"] = round(float(part["left_hand_detection_rate"].mean()), 4)
        row[f"{split}_right_hand_rate"] = round(float(part["right_hand_detection_rate"].mean()), 4)
        # The quantity the pairing objective actually balanced.
        row[f"{split}_hand_quality"] = round(float(part["hand_quality"].mean()), 4)
    summary_rows.append(row)

fold_summary = pd.DataFrame(summary_rows)
print("FOLD SUMMARY")
print(fold_summary[["fold", "test_signers", "validation_signers", "train_clips",
                    "validation_clips", "test_clips", "test_mean_frames",
                    "test_hand_quality"]].to_string(index=False))

print("\nTest-set balance across folds")
for column in ("test_clips", "test_mean_frames", "test_hand_quality"):
    values = fold_summary[column]
    print(f"  {column:<22} {values.min()} to {values.max()} (std {values.std():.3f})")

# ---- Signer-overlap control: ONE fixed split, deliberately invalid -------
# A single stratified clip-level split that ignores signer, so the same person
# appears in train and test. NOT an evaluation of the models.
#
# Only ONE control split, not one per fold: eight would mean 3 models x 2
# branches x 8 = 48 extra runs to establish a single contrast; one gives the
# same contrast in 6.
#
# Its split SIZES are matched to the average primary fold as closely as the
# class sizes allow, so that the comparison is not confounded by training-set
# size. Any residual difference is recorded in the manifest.
#
# Naming matters. The difference between this and the signer-independent result
# is the "signer-overlap performance gap". Identity memorisation is the likely
# explanation, but the gap also absorbs recording-session and background
# similarity, so it is reported as a gap and discussed as probably reflecting
# memorisation - not asserted to BE identity leakage.

def allocate_stratified(class_sizes: Dict[int, int], targets: Dict[str, int],
                        splits: Tuple[str, ...] = ("train", "validation", "test"),
                        minimum_per_split: int = 1) -> Tuple[Dict[int, Dict[str, int]], Dict[str, int]]:
    """Per-class counts summing to each class size and to the global targets.

    Largest-remainder within each class guarantees the per-class totals; a
    deterministic repair pass then moves single units between splits until the
    global totals match, never dropping a class below the minimum for a split.
    """
    total = sum(class_sizes.values())
    shares = {split: targets[split] / total for split in splits}
    counts: Dict[int, Dict[str, int]] = {}

    for label_id in sorted(class_sizes):
        size = class_sizes[label_id]
        ideal = {split: size * shares[split] for split in splits}
        allocation = {split: int(ideal[split]) for split in splits}
        remainder = size - sum(allocation.values())
        order = sorted(splits, key=lambda s: (-(ideal[s] - int(ideal[s])), splits.index(s)))
        for index in range(remainder):
            allocation[order[index % len(order)]] += 1

        for split in splits:                       # guarantee class coverage
            while allocation[split] < minimum_per_split:
                donor = max(splits, key=lambda s: (allocation[s], -splits.index(s)))
                if allocation[donor] <= minimum_per_split:
                    break
                allocation[donor] -= 1
                allocation[split] += 1
        counts[label_id] = allocation

    def totals() -> Dict[str, int]:
        return {split: sum(counts[c][split] for c in counts) for split in splits}

    # Repair with a rotating cursor. Always restarting from class 0 would drain
    # the first few classes down to the minimum while leaving the rest
    # untouched; rotating spreads the adjustment evenly across the label space.
    labels = sorted(class_sizes)
    cursor = 0
    for _ in range(20 * len(labels)):
        current = totals()
        if current == targets:
            break
        over = max(splits, key=lambda s: (current[s] - targets[s], -splits.index(s)))
        under = min(splits, key=lambda s: (current[s] - targets[s], splits.index(s)))
        if current[over] <= targets[over] or current[under] >= targets[under]:
            break
        moved = False
        for step in range(len(labels)):
            label_id = labels[(cursor + step) % len(labels)]
            if counts[label_id][over] > minimum_per_split:
                counts[label_id][over] -= 1
                counts[label_id][under] += 1
                cursor = (cursor + step + 1) % len(labels)
                moved = True
                break
        if not moved:
            break

    return counts, totals()


CONTROL_TARGETS = {
    "train": int(round(fold_summary["train_clips"].mean())),
    "validation": int(round(fold_summary["validation_clips"].mean())),
    "test": int(round(fold_summary["test_clips"].mean())),
}
# Force the targets to sum to the dataset size without disturbing the two
# smaller splits, which are the ones the comparison is sensitive to.
CONTROL_TARGETS["train"] = len(manifest) - CONTROL_TARGETS["validation"] - CONTROL_TARGETS["test"]

class_sizes = manifest.groupby("label_id").size().to_dict()
control_counts, control_achieved = allocate_stratified(class_sizes, CONTROL_TARGETS)
control_deviation = {split: control_achieved[split] - CONTROL_TARGETS[split]
                     for split in CONTROL_TARGETS}

print("\nSIGNER-OVERLAP CONTROL (one fixed split, invalid by design)")
print(f"  primary fold averages : train {fold_summary['train_clips'].mean():.1f}, "
      f"validation {fold_summary['validation_clips'].mean():.1f}, "
      f"test {fold_summary['test_clips'].mean():.1f}")
print(f"  control targets       : {CONTROL_TARGETS}")
print(f"  control achieved      : {control_achieved}")
print(f"  deviation             : {control_deviation}")
if any(control_deviation.values()):
    print("  NOTE: exact equality was not reachable given the per-class sizes. The "
          "deviation is recorded in the manifest so later analysis does not "
          "attribute the whole control-vs-primary gap to signer overlap alone.")

control_generator = np.random.default_rng(RANDOM_SEED)
control_split = np.empty(len(manifest), dtype=object)
for label_id, group in manifest.groupby("label_id").groups.items():
    indices = np.array(sorted(group))
    control_generator.shuffle(indices)
    allocation = control_counts[label_id]
    cut_train = allocation["train"]
    cut_validation = cut_train + allocation["validation"]
    control_split[indices[:cut_train]] = "train"
    control_split[indices[cut_train:cut_validation]] = "validation"
    control_split[indices[cut_validation:]] = "test"

control_assignments = pd.DataFrame({
    "clip_id": manifest["clip_id"], "signer_id": manifest["signer_id"],
    "label_id": manifest["label_id"], "label": manifest["label"],
    "split": control_split, "split_type": "signer_overlap_INVALID_control",
})

control_train_signers = set(control_assignments.loc[
    control_assignments["split"] == "train", "signer_id"])
control_test_signers = set(control_assignments.loc[
    control_assignments["split"] == "test", "signer_id"])
overlap = control_train_signers & control_test_signers
control_classes = {split: int(control_assignments.loc[
    control_assignments["split"] == split, "label_id"].nunique())
    for split in ("train", "validation", "test")}

print(f"  classes per split     : {control_classes}")
print(f"  signers shared train/test: {len(overlap)} of {len(all_signers)}")
print("  Overlap is intentional. Without it the control would measure nothing.")

if not overlap:
    raise RuntimeError("The control split has no signer overlap; it would measure nothing.")
if any(count != EXPECTED_LABELS for count in control_classes.values()):
    raise RuntimeError(f"Control split does not cover all classes: {control_classes}")
if any(abs(value) > 0.02 * len(manifest) for value in control_deviation.values()):
    raise RuntimeError(f"Control sizes deviate too far from the primary folds: "
                       f"{control_deviation}")

# ---- Plots ----------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight", "font.size": 9})
plot_paths: List[str] = []


def save(figure, name: str) -> None:
    path = PLOT_DIR / name
    figure.savefig(path)
    plt.close(figure)
    plot_paths.append(str(path))
    print(f"  {name}")


figure, axes = plt.subplots(1, 2, figsize=(11, 4))
labels_for_pairs = [f"{r.signer_a}+{r.signer_b}" for r in pair_table.itertuples()]
axes[0].bar(range(len(pair_table)), pair_table["mean_frames"], color="#4C72B0")
axes[0].axhline(signer_table["mean_frames"].mean(), color="#C44E52", linestyle="--",
                label="dataset mean")
axes[0].scatter(np.repeat(range(len(pair_table)), 2),
                [lookup.loc[s, "mean_frames"] for pair in SIGNER_PAIRS_LIST for s in pair],
                color="#DD8452", zorder=3, s=18, label="individual signers")
axes[0].set_xticks(range(len(pair_table)))
axes[0].set_xticklabels(labels_for_pairs, rotation=45, ha="right")
axes[0].set_ylabel("mean frames"); axes[0].set_title("Clip length: pairs vs individuals")
axes[0].legend(fontsize=7)

axes[1].bar(range(len(pair_table)), pair_table["hand_quality"], color="#55A868")
axes[1].axhline(signer_table["hand_quality"].mean(), color="#C44E52", linestyle="--")
axes[1].scatter(np.repeat(range(len(pair_table)), 2),
                [lookup.loc[s, "hand_quality"] for pair in SIGNER_PAIRS_LIST for s in pair],
                color="#DD8452", zorder=3, s=18)
axes[1].set_xticks(range(len(pair_table)))
axes[1].set_xticklabels(labels_for_pairs, rotation=45, ha="right")
axes[1].set_ylabel("hand quality  (left + right) / 2")
axes[1].set_title("Hand quality: pairs vs individuals")
save(figure, "pair_balance.png")

matrix = np.zeros((len(all_signers), N_FOLDS))
for definition in fold_definitions:
    for signer in definition["train_signers"]:
        matrix[all_signers.index(signer), definition["fold"]] = 0
    for signer in definition["validation_signers"]:
        matrix[all_signers.index(signer), definition["fold"]] = 1
    for signer in definition["test_signers"]:
        matrix[all_signers.index(signer), definition["fold"]] = 2

figure, axis = plt.subplots(figsize=(7, 5))
axis.imshow(matrix, cmap=matplotlib.colors.ListedColormap(["#DDE3EA", "#DD8452", "#C44E52"]),
            aspect="auto", vmin=0, vmax=2)
axis.set_xticks(range(N_FOLDS)); axis.set_xticklabels([f"fold {f}" for f in range(N_FOLDS)],
                                                      rotation=45, ha="right")
axis.set_yticks(range(len(all_signers))); axis.set_yticklabels(all_signers)
axis.set_title("Fold membership: grey = train, orange = validation, red = test")
for spine in axis.spines.values():
    spine.set_visible(False)
save(figure, "fold_membership.png")

In [ ]:
# ============================================================
# CELL 7 — Freeze, hash and write the manifest
# ============================================================

def canonical_hash(frame: pd.DataFrame, columns: Sequence[str]) -> str:
    ordered = frame.sort_values(list(columns), kind="mergesort")[list(columns)]
    lines = ["\t".join(f"{column}={record[column]}" for column in columns)
             for record in ordered.to_dict("records")]
    return hashlib.sha256(("\n".join(lines) + "\n").encode("utf-8")).hexdigest()[:16]


FOLD_HASH = canonical_hash(fold_assignments, ["fold", "clip_id", "split"])
MANIFEST_HASH = canonical_hash(
    manifest, ["clip_id", "label", "label_id", "signer_id", "session",
               "relative_path", "original_frames"])
PAIRS_HASH = hashlib.sha256(
    json.dumps([list(pair) for pair in SIGNER_PAIRS_LIST], separators=(",", ":")).encode()
).hexdigest()[:16]
PAIR_ORDER_HASH = hashlib.sha256(
    json.dumps(PAIR_ORDER, separators=(",", ":")).encode()).hexdigest()[:16]
# The control split drives six later runs, so it is frozen and hashed too.
CONTROL_HASH = canonical_hash(control_assignments, ["clip_id", "split"])
LABEL_MAP_HASH = hashlib.sha256(
    json.dumps(LABEL_TO_ID, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()[:16]

# One id binding every frozen component together. Defined as a function so the
# same code can recompute it from the RELOADED manifest below; a second separate
# expression would only prove that the two expressions agree with each other.
EXPERIMENT_ID_COMPONENTS = ("clip_table", "manifest", "fold_assignments",
                            "signer_pairs", "pair_order", "control_split",
                            "label_map", "manifest_version")


def compute_experiment_id(components: Dict[str, Any]) -> str:
    return hashlib.sha256(
        "|".join(str(components[name]) for name in EXPERIMENT_ID_COMPONENTS).encode()
    ).hexdigest()[:16]


EXPERIMENT_ID_INPUTS = {
    "clip_table": CLIP_TABLE_HASH, "manifest": MANIFEST_HASH,
    "fold_assignments": FOLD_HASH, "signer_pairs": PAIRS_HASH,
    "pair_order": PAIR_ORDER_HASH, "control_split": CONTROL_HASH,
    "label_map": LABEL_MAP_HASH, "manifest_version": MANIFEST_VERSION,
}
EXPERIMENT_ID = compute_experiment_id(EXPERIMENT_ID_INPUTS)

atomic_write(MASTER_MANIFEST, lambda h: manifest.to_csv(h, index=False))
atomic_write(FOLD_ASSIGNMENTS, lambda h: fold_assignments.to_csv(h, index=False))
atomic_write(FOLD_SUMMARY, lambda h: fold_summary.to_csv(h, index=False))
atomic_write(CONTROL_ASSIGNMENTS, lambda h: control_assignments.to_csv(h, index=False))
atomic_write(LABEL_MAP, lambda h: h.write(json.dumps(
    {"label_to_id": LABEL_TO_ID, "id_to_label": {str(k): v for k, v in ID_TO_LABEL.items()},
     "n_classes": len(LABEL_TO_ID), "hash": LABEL_MAP_HASH}, indent=2).encode()))
atomic_write(SIGNER_PAIRS, lambda h: h.write(json.dumps({
    "pairs": [{"pair_index": index, "signers": list(pair)}
              for index, pair in enumerate(SIGNER_PAIRS_LIST)],
    "balance_weights": BALANCE_WEIGHTS,
    "reporting_features": REPORTING_FEATURES,
    "forbidden_pairs": [sorted(pair) for pair in FORBIDDEN_PAIRS],
    "pair_statistics": pair_table.to_dict("records"),
    "method": "exact minimum-variance perfect matching (bitmask DP over 2^16 states); "
              "cost is the squared deviation of each pair's summed z-scores from zero",
    "hash": PAIRS_HASH,
}, indent=2, default=str).encode()))

experiment = {
    "experiment_id": EXPERIMENT_ID, "manifest_version": MANIFEST_VERSION,
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "environment": ENVIRONMENT,
    "provenance": {
        "pipeline_id": PIPELINE_ID, "audit_verdict": AUDIT_VERDICT, "audit_mode": AUDIT_MODE,
        "clip_table_hash": CLIP_TABLE_HASH,
        "audit_summary": str(AUDIT_JSON_PATH),
        "signer_statistics": {b: str(p) for b, p in STATS_PATHS.items()},
        "signer_statistics_provenance": STATS_PROVENANCE,
        "branch_roots": {b: str(p) for b, p in BRANCH_ROOTS.items()},
    },
    "hashes": {"manifest": MANIFEST_HASH, "fold_assignments": FOLD_HASH,
               "signer_pairs": PAIRS_HASH, "pair_order": PAIR_ORDER_HASH,
               "control_split": CONTROL_HASH, "label_map": LABEL_MAP_HASH},
    "pairing": {
        "unordered_optimal_pairs": [list(pair) for pair in SIGNER_PAIRS_LIST],
        "final_cyclic_order": PAIR_ORDER,
        "final_cyclic_pairs": [list(SIGNER_PAIRS_LIST[index]) for index in PAIR_ORDER],
        "order_optimisation": "exhaustive search over all (n-1)! cyclic orders with "
                              "pair 0 pinned, minimising the variance of per-fold "
                              "training clip counts; ties broken lexicographically. "
                              "Pair membership is unchanged by this step.",
        "baseline_train_sizes": [int(v) for v in baseline_train],
        "optimised_train_sizes": [int(v) for v in optimised_train],
        "train_size_spread": TRAIN_SPREAD,
    },
    "fold_clip_counts": fold_clip_counts.to_dict("records"),
    "design": {
        "n_folds": N_FOLDS, "train_signers": EXPECTED_SIGNERS - 4,
        "validation_signers": VALIDATION_SIGNERS_PER_FOLD,
        "test_signers": TEST_SIGNERS_PER_FOLD,
        "rotation": "fold i tests final_cyclic_pairs[i] and validates "
                    "final_cyclic_pairs[(i+1) mod n_folds]",
        "balance_weights": BALANCE_WEIGHTS,
    "reporting_features": REPORTING_FEATURES,
        "forbidden_pairs": [sorted(p) for p in FORBIDDEN_PAIRS],
        "pooled_out_of_fold_predictions": int(len(pooled)),
    },
    "statistical_plan": {
        "primary_performance": "pooled 6,000 out-of-fold predictions",
        "variability": "mean and standard deviation across the 8 folds",
        "primary_inference": "exhaustive paired sign-flip (permutation) test over all "
                             "2^8 sign assignments of the 8 fold-level differences. "
                             "Pre-committed as THE primary test. The enumeration is "
                             "exhaustive, but the folds are NOT independent - they "
                             "share many training signers - so the test is not exact "
                             "in the strict sense. With n=8 the minimum attainable "
                             "two-sided p is 2/256 = 0.0078, so power is limited and "
                             "effect sizes with intervals are reported regardless of "
                             "significance.",
        "reporting_wording": "An exhaustive paired sign-flip test across the eight "
                             "fold-level differences was used as the pre-specified "
                             "primary inference; results are interpreted cautiously "
                             "because cross-validation training sets overlap.",
        "sensitivity_analysis": "fold-clustered bootstrap for confidence intervals, "
                                "reported alongside the primary test as sensitivity "
                                "analysis - never as a substitute if the primary "
                                "test is not significant",
        "signer_analysis": "16 paired signer differences, EXPLORATORY only - the two test "
                           "signers in a fold share one trained model and folds share "
                           "training signers, so they are not independent",
        "per_class_metrics": "computed on the pooled predictions, not per fold, because a "
                             "single fold gives only ~3 examples per class",
        "random_forest_caveat": "the full branch yields roughly 2.8x more RF features than "
                                "the reduced branch, so an RF difference partly reflects "
                                "dimensionality rather than facial information",
        "neural_caveat": "the recurrent core is matched across branches, but the input "
                         "projection differs (1759x256 vs 631x256, about 289k extra "
                         "weights), so total parameter counts are not identical and are "
                         "reported per model",
    },
    "counts": {"clips": len(manifest), "labels": len(LABEL_TO_ID),
               "signers": len(all_signers), "fold_assignment_rows": len(fold_assignments)},
    "folds": [{**definition, "train_signers": definition["train_signers"],
               "validation_signers": definition["validation_signers"],
               "test_signers": definition["test_signers"]}
              for definition in fold_definitions],
    "fold_summary": fold_summary.to_dict("records"),
    "signer_properties": signer_table.to_dict("records"),
    "control_split": {
        "file": str(CONTROL_ASSIGNMENTS), "type": "signer_overlap_INVALID_control",
        "n_splits": 1,
        "purpose": "establishes the signer-overlap performance gap: the difference "
                   "between this and the signer-independent result. Identity "
                   "memorisation is the likely explanation, but the gap also absorbs "
                   "session and background similarity, so it is reported as a gap "
                   "rather than asserted to be identity leakage.",
        "runs_required": "3 models x 2 branches = 6 (not 48)",
        "target_counts": CONTROL_TARGETS, "achieved_counts": control_achieved,
        "count_deviation": control_deviation,
        "primary_fold_average_counts": {
            "train": float(fold_summary["train_clips"].mean()),
            "validation": float(fold_summary["validation_clips"].mean()),
            "test": float(fold_summary["test_clips"].mean())},
        "size_matching_note": "control split sizes are matched to the average primary "
                              "fold so the comparison is not confounded by training-set "
                              "size; any residual deviation is recorded above",
        "hash": CONTROL_HASH,
        "seed": RANDOM_SEED, "signers_shared_train_test": len(overlap),
    },
    "path_portability": {
        "warning": "full_path and reduced_path in master_manifest.csv are absolute "
                   "paths from THIS runtime and will not be valid in another Kaggle "
                   "session. Downstream notebooks must rediscover the attached "
                   "Phase 5 roots and rebuild absolute paths from relative_path.",
        "authoritative_field": "relative_path",
        "rediscover_in_each_session": [
            "the attached Phase 5 'full' root (…/normalized_landmarks/full)",
            "the attached Phase 5 'reduced' root (…/normalized_landmarks/reduced)",
            "the notebook 6 audit summary (post_normalization_audit.json)",
            "the notebook 6 statistics (signer_feature_stats_{full,reduced}.npz)",
        ],
        "stale_absolute_paths": ["master_manifest.csv:full_path",
                                 "master_manifest.csv:reduced_path",
                                 "provenance.audit_summary",
                                 "provenance.signer_statistics.*",
                                 "provenance.branch_roots.*"],
        "verify_before_loading": ["experiment_id", "manifest_version",
                                  "hashes.manifest", "hashes.fold_assignments",
                                  "hashes.signer_pairs", "hashes.pair_order",
                                  "hashes.control_split", "hashes.label_map",
                                  "provenance.clip_table_hash",
                                  "provenance.pipeline_id"],
        "experiment_id_recipe": "sha256 of the '|'-joined values of "
                                "EXPERIMENT_ID_COMPONENTS in that order, truncated to "
                                "16 hex characters; recompute it to confirm no "
                                "recorded hash was edited",
    },
    "outputs": {"master_manifest": str(MASTER_MANIFEST),
                "fold_assignments": str(FOLD_ASSIGNMENTS),
                "signer_pairs": str(SIGNER_PAIRS), "label_map": str(LABEL_MAP),
                "fold_summary": str(FOLD_SUMMARY), "control": str(CONTROL_ASSIGNMENTS),
                "plots": plot_paths},
}
atomic_write(EXPERIMENT_MANIFEST,
             lambda h: h.write(json.dumps(experiment, indent=2, default=str).encode()))

# Reload every frozen artefact and re-hash it. Hashing what is in memory only
# proves the computation was consistent; hashing what came back off disk proves
# the files the next six notebooks will read are the ones verified here.
reloaded_manifest = pd.read_csv(MASTER_MANIFEST)
reloaded_folds = pd.read_csv(FOLD_ASSIGNMENTS)
reloaded_control = pd.read_csv(CONTROL_ASSIGNMENTS)
reloaded_experiment = json.loads(EXPERIMENT_MANIFEST.read_text(encoding="utf-8"))
reloaded_label_map = json.loads(LABEL_MAP.read_text(encoding="utf-8"))
reloaded_signer_pairs = json.loads(SIGNER_PAIRS.read_text(encoding="utf-8"))

round_trip_manifest = canonical_hash(
    reloaded_manifest, ["clip_id", "label", "label_id", "signer_id", "session",
                        "relative_path", "original_frames"])
round_trip_folds = canonical_hash(reloaded_folds, ["fold", "clip_id", "split"])
round_trip_control = canonical_hash(reloaded_control, ["clip_id", "split"])
round_trip_order = hashlib.sha256(
    json.dumps(reloaded_experiment["pairing"]["final_cyclic_order"],
               separators=(",", ":")).encode()).hexdigest()[:16]
# The label map decides which integer means which Bangla sign, so a silent
# change here would relabel every prediction in the thesis.
round_trip_label_map = hashlib.sha256(
    json.dumps({key: int(value)
                for key, value in reloaded_label_map["label_to_id"].items()},
               sort_keys=True, separators=(",", ":")).encode()).hexdigest()[:16]
round_trip_pairs = hashlib.sha256(
    json.dumps([list(entry["signers"]) for entry in reloaded_signer_pairs["pairs"]],
               separators=(",", ":")).encode()).hexdigest()[:16]

print("=" * 78)
print(f"EXPERIMENT MANIFEST FROZEN  ({MANIFEST_VERSION})")
print("=" * 78)
print(f"  experiment id        : {EXPERIMENT_ID}")
print(f"  manifest hash        : {MANIFEST_HASH}  round trip {round_trip_manifest}")
print(f"  fold hash            : {FOLD_HASH}  round trip {round_trip_folds}")
print(f"  signer pairs hash    : {PAIRS_HASH}  round trip {round_trip_pairs}")
print(f"  pair order hash      : {PAIR_ORDER_HASH}  round trip {round_trip_order}")
print(f"  control split hash   : {CONTROL_HASH}  round trip {round_trip_control}")
print(f"  label map hash       : {LABEL_MAP_HASH}  round trip {round_trip_label_map}")
print(f"  clip table hash      : {CLIP_TABLE_HASH}  (from notebook 6)")
print(f"  pipeline id          : {PIPELINE_ID}")

# The files are fingerprinted, but experiment_manifest.json also RECORDS those
# fingerprints and the experiment id. Nothing above would notice if those
# recorded values were edited while the files themselves stayed intact, so the
# manifest is now checked against what was verified here.
stored_hashes = reloaded_experiment.get("hashes", {})
stored_provenance = reloaded_experiment.get("provenance", {})
manifest_field_failures = [
    f"{field}: stored {stored!r}, expected {expected!r}"
    for field, stored, expected in (
        ("experiment_id", reloaded_experiment.get("experiment_id"), EXPERIMENT_ID),
        ("manifest_version", reloaded_experiment.get("manifest_version"), MANIFEST_VERSION),
        ("hashes.manifest", stored_hashes.get("manifest"), MANIFEST_HASH),
        ("hashes.fold_assignments", stored_hashes.get("fold_assignments"), FOLD_HASH),
        ("hashes.signer_pairs", stored_hashes.get("signer_pairs"), PAIRS_HASH),
        ("hashes.pair_order", stored_hashes.get("pair_order"), PAIR_ORDER_HASH),
        ("hashes.control_split", stored_hashes.get("control_split"), CONTROL_HASH),
        ("hashes.label_map", stored_hashes.get("label_map"), LABEL_MAP_HASH),
        ("provenance.clip_table_hash", stored_provenance.get("clip_table_hash"),
         CLIP_TABLE_HASH),
        ("provenance.pipeline_id", stored_provenance.get("pipeline_id"), PIPELINE_ID),
    ) if stored != expected
]
if manifest_field_failures:
    raise RuntimeError("experiment_manifest.json records values that disagree with "
                       "what was verified: " + "; ".join(manifest_field_failures))

# Recompute the experiment id purely from the reloaded components. Altering any
# single recorded hash stops this reproducing the stored id.
recomputed_experiment_id = compute_experiment_id({
    "clip_table": stored_provenance.get("clip_table_hash"),
    "manifest": stored_hashes.get("manifest"),
    "fold_assignments": stored_hashes.get("fold_assignments"),
    "signer_pairs": stored_hashes.get("signer_pairs"),
    "pair_order": stored_hashes.get("pair_order"),
    "control_split": stored_hashes.get("control_split"),
    "label_map": stored_hashes.get("label_map"),
    "manifest_version": reloaded_experiment.get("manifest_version"),
})
if recomputed_experiment_id != reloaded_experiment.get("experiment_id"):
    raise RuntimeError(
        f"The stored experiment id {reloaded_experiment.get('experiment_id')} is not "
        f"reproducible from the stored components ({recomputed_experiment_id}).")
print(f"  experiment id recomputed from the reloaded manifest: {recomputed_experiment_id}")

round_trip_failures = [
    name for name, expected, actual in (
        ("master_manifest.csv", MANIFEST_HASH, round_trip_manifest),
        ("fold_assignments.csv", FOLD_HASH, round_trip_folds),
        ("pair order in experiment_manifest.json", PAIR_ORDER_HASH, round_trip_order),
        ("CONTROL_signer_overlap_split.csv", CONTROL_HASH, round_trip_control),
        ("label_map.json", LABEL_MAP_HASH, round_trip_label_map),
        ("signer_pairs.json", PAIRS_HASH, round_trip_pairs),
    ) if expected != actual
]
if round_trip_failures:
    raise RuntimeError(
        "These files on disk do not reproduce their verified hashes: "
        + ", ".join(round_trip_failures))
print("\n  all six frozen artefacts round-tripped correctly")

print("\n  Files written:")
for path in (MASTER_MANIFEST, FOLD_ASSIGNMENTS, FOLD_SUMMARY, SIGNER_PAIRS,
             LABEL_MAP, EXPERIMENT_MANIFEST, CONTROL_ASSIGNMENTS):
    print(f"    {path.name}")
for path in plot_paths:
    print(f"    fold_plots/{Path(path).name}")

print("\n" + "=" * 78)
print("NOTEBOOK 7 COMPLETE")
print("=" * 78)
print(f"  Quote experiment id {EXPERIMENT_ID} in every downstream notebook.")
print("  Notebooks 8 to 13 must read these folds and never create their own split.")